# MLB Team Wins Prediction Notebook

This notebook builds **three separate, independently runnable model sections** to predict a team's season wins (`W`) from team performance statistics.

## Data files expected in the same working directory
- `data.csv` — training data
- `predict.csv` — scoring data
- `sample_submission.csv` — output template
- `environment.yml` — supplied runtime spec

## Important execution note
The supplied `environment.yml` includes core scientific Python packages such as `python=3.11.5`, `pandas=2.1.1`, `numpy=1.26.0`, `scikit-learn=1.3.0`, and `lightgbm=4.1.0`, but it **does not include `auto-sklearn`**.  
Accordingly:

- **Section 1** uses `auto-sklearn` **if it is installed** in the active environment.
- If `auto-sklearn` is unavailable, that section exits cleanly with an explicit message instead of failing the entire notebook.
- **Sections 2 and 3** run with the supplied environment as-is.

Each section:
1. Loads the data independently.
2. Rebuilds the feature engineering pipeline independently.
3. Trains its own model(s).
4. Writes its own prediction file:
   - `prediction_submission_1.csv`
   - `prediction_submission_2.csv`
   - `prediction_submission_3.csv`

## Modeling design
To improve cross-era robustness, the engineered features combine:
- raw season totals,
- rate stats,
- simplified sabermetric features derivable from the provided data,
- year-relative normalization features,
- decade indicators already present in the dataset.

## Hyperparameters
Each model section contains a **clearly labeled configuration block** showing the main hyperparameters you can adjust.

In [ ]:
# ============================================================
# COMMON UTILITIES
# Re-run this cell before running any model section.
# ============================================================

from __future__ import annotations

import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "data.csv"
PREDICT_PATH = DATA_DIR / "predict.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

def validate_input_files() -> None:
    required = [TRAIN_PATH, PREDICT_PATH, SUBMISSION_PATH]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required files: " + ", ".join(missing)
        )

def load_base_data():
    validate_input_files()
    train_df = pd.read_csv(TRAIN_PATH)
    pred_df = pd.read_csv(PREDICT_PATH)
    submission_df = pd.read_csv(SUBMISSION_PATH)
    return train_df, pred_df, submission_df

def _safe_divide(a, b):
    a = pd.Series(a)
    b = pd.Series(b)
    out = np.where(pd.to_numeric(b, errors="coerce") == 0, np.nan, a / b)
    return pd.Series(out, index=a.index)

def add_engineered_features(train_df: pd.DataFrame, pred_df: pd.DataFrame):
    '''
    Create era-aware team features using only variables available in the uploaded files.
    The function returns transformed copies of train and prediction data.
    '''
    train = train_df.copy()
    pred = pred_df.copy()

    combined = pd.concat(
        [train.assign(_dataset="train"), pred.assign(_dataset="predict")],
        axis=0,
        ignore_index=True,
        sort=False
    )

    # Standard inning conversion
    combined["IP"] = combined["IPouts"] / 3.0

    # --- Offensive metrics derivable from supplied data ---
    combined["TB"] = (
        combined["H"]
        + combined["2B"]
        + 2 * combined["3B"]
        + 3 * combined["HR"]
    )
    combined["OBP_simplified"] = _safe_divide(combined["H"] + combined["BB"], combined["AB"] + combined["BB"])
    combined["SLG"] = _safe_divide(combined["TB"], combined["AB"])
    combined["OPS"] = combined["OBP_simplified"] + combined["SLG"]
    combined["RunsCreated_basic"] = _safe_divide(
        (combined["H"] + combined["BB"]) * combined["TB"],
        (combined["AB"] + combined["BB"])
    )
    combined["PA_approx"] = combined["AB"] + combined["BB"]
    combined["K_rate_approx"] = _safe_divide(combined["SO"], combined["PA_approx"])
    combined["BB_rate_approx"] = _safe_divide(combined["BB"], combined["PA_approx"])
    combined["HR_rate_approx"] = _safe_divide(combined["HR"], combined["PA_approx"])
    combined["SB_rate_approx"] = _safe_divide(combined["SB"], combined["PA_approx"])

    # --- Pitching metrics derivable from supplied data ---
    combined["WHIP_calc"] = _safe_divide(combined["BBA"] + combined["HA"], combined["IP"])
    combined["H_per_9"] = _safe_divide(combined["HA"] * 9.0, combined["IP"])
    combined["BB_per_9"] = _safe_divide(combined["BBA"] * 9.0, combined["IP"])
    combined["SOA_per_9"] = _safe_divide(combined["SOA"] * 9.0, combined["IP"])
    combined["HRA_per_9"] = _safe_divide(combined["HRA"] * 9.0, combined["IP"])

    # --- Defensive / team context ---
    combined["FieldingPct"] = combined["FP"]
    combined["DP_per_game"] = _safe_divide(combined["DP"], combined["G"])
    combined["Errors_per_game"] = _safe_divide(combined["E"], combined["G"])

    # --- Team quality / context features ---
    combined["R_per_game"] = _safe_divide(combined["R"], combined["G"])
    combined["RA_per_game"] = _safe_divide(combined["RA"], combined["G"])
    combined["RunDiff"] = combined["R"] - combined["RA"]
    combined["RunDiff_per_game"] = _safe_divide(combined["RunDiff"], combined["G"])
    combined["PythagWinPct"] = _safe_divide(
        combined["R"] ** 2,
        (combined["R"] ** 2) + (combined["RA"] ** 2)
    )
    combined["WinPct_proxy"] = _safe_divide(combined.get("W", np.nan), combined["G"])

    # --- Era-aware normalization by season ---
    season_base_cols = [
        "R_per_game", "RA_per_game", "RunDiff_per_game", "OBP_simplified", "SLG",
        "OPS", "RunsCreated_basic", "K_rate_approx", "BB_rate_approx",
        "HR_rate_approx", "SB_rate_approx", "WHIP_calc", "H_per_9", "BB_per_9",
        "SOA_per_9", "HRA_per_9", "FieldingPct", "DP_per_game", "Errors_per_game",
        "PythagWinPct", "ERA", "SV"
    ]

    season_means = combined.groupby("yearID")[season_base_cols].transform("mean")
    season_stds = combined.groupby("yearID")[season_base_cols].transform("std").replace(0, np.nan)

    for col in season_base_cols:
        combined[f"{col}_year_ratio"] = _safe_divide(combined[col], season_means[col])
        combined[f"{col}_year_z"] = _safe_divide(combined[col] - season_means[col], season_stds[col])

    # Preserve only model-usable fields plus identifiers
    train_out = combined.loc[combined["_dataset"] == "train"].drop(columns=["_dataset"])
    pred_out = combined.loc[combined["_dataset"] == "predict"].drop(columns=["_dataset"])

    return train_out, pred_out

def build_feature_lists(train_df: pd.DataFrame):
    '''
    Return numeric and categorical feature lists after excluding leakage / identifiers.
    '''
    target = "W"
    exclude_cols = {
        target,
        "ID",
        "teamID",
        "year_label",
        "decade_label",
        "win_bins",
        "WinPct_proxy",   # target-derived, so exclude
    }

    categorical_cols = []
    if "franchID" in train_df.columns:
        categorical_cols.append("franchID")

    # Keep yearID because baseball run environment changes across eras.
    numeric_cols = [
        c for c in train_df.columns
        if c not in exclude_cols | set(categorical_cols)
        and pd.api.types.is_numeric_dtype(train_df[c])
    ]

    return numeric_cols, categorical_cols

def build_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return preprocessor

def get_xy(train_df: pd.DataFrame, pred_df: pd.DataFrame):
    numeric_cols, categorical_cols = build_feature_lists(train_df)
    feature_cols = numeric_cols + categorical_cols

    X_train = train_df[feature_cols].copy()
    y_train = train_df["W"].copy()
    X_pred = pred_df[feature_cols].copy()

    return X_train, y_train, X_pred, numeric_cols, categorical_cols

def evaluate_model_cv(estimator, X, y, groups=None, n_splits=5):
    '''
    Returns positive MAE values (lower is better).
    '''
    if groups is not None:
        cv = GroupKFold(n_splits=n_splits)
        scores = cross_val_score(
            estimator, X, y,
            cv=cv.split(X, y, groups=groups),
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )
    else:
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
        scores = cross_val_score(
            estimator, X, y,
            cv=cv,
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )
    return -scores

def write_submission(pred_values, submission_template, out_path):
    sub = submission_template.copy()
    sub["W"] = np.clip(np.rint(pred_values), 0, 162).astype(int)
    sub.to_csv(out_path, index=False)
    return sub

def prepare_datasets():
    train_raw, pred_raw, submission = load_base_data()
    train_fe, pred_fe = add_engineered_features(train_raw, pred_raw)
    X_train, y_train, X_pred, numeric_cols, categorical_cols = get_xy(train_fe, pred_fe)
    groups = train_fe["yearID"]
    return {
        "train_raw": train_raw,
        "pred_raw": pred_raw,
        "train_fe": train_fe,
        "pred_fe": pred_fe,
        "X_train": X_train,
        "y_train": y_train,
        "X_pred": X_pred,
        "submission": submission,
        "groups": groups,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
    }

print("Utilities loaded. Ready to run any section.")

## Section 1 — Model 1: `auto-sklearn`

This section uses `AutoSklearnRegressor` when the package is available.

### Adjustable hyperparameters
Edit the config block in the next cell:
- `AUTO_SKLEARN_TIME_LEFT_FOR_THIS_TASK` — total search budget in seconds.
- `AUTO_SKLEARN_PER_RUN_TIME_LIMIT` — cap per model training attempt.
- `AUTO_SKLEARN_MEMORY_LIMIT_MB` — memory budget.
- `AUTO_SKLEARN_ENSEMBLE_SIZE` — ensemble size.
- `AUTO_SKLEARN_RESAMPLING_FOLDS` — CV folds used by auto-sklearn.
- `AUTO_SKLEARN_N_JOBS` — parallel workers.

### Why this matters
A **hyperparameter** is a setting chosen before training starts.  
Real-life metaphor: it is like deciding the oven temperature and bake time before making bread. Those settings strongly affect the final result.

In [ ]:
# ============================================================
# MODEL 1 CONFIGURATION — ADJUST THESE HYPERPARAMETERS
# ============================================================

AUTO_SKLEARN_TIME_LEFT_FOR_THIS_TASK = 900      # total search time budget in seconds
AUTO_SKLEARN_PER_RUN_TIME_LIMIT = 120           # max seconds per candidate model
AUTO_SKLEARN_MEMORY_LIMIT_MB = 4096             # memory budget for auto-sklearn
AUTO_SKLEARN_ENSEMBLE_SIZE = 50                 # larger can improve quality but costs time
AUTO_SKLEARN_RESAMPLING_FOLDS = 5               # cross-validation folds inside auto-sklearn
AUTO_SKLEARN_N_JOBS = 1                         # set >1 only if your environment supports it
AUTO_SKLEARN_TMP_FOLDER = "autosklearn_tmp"
AUTO_SKLEARN_OUTPUT_FILE = "prediction_submission_1.csv"

In [ ]:
# ============================================================
# MODEL 1 EXECUTION
# ============================================================

data_bundle = prepare_datasets()
X_train = data_bundle["X_train"]
y_train = data_bundle["y_train"]
X_pred = data_bundle["X_pred"]
submission = data_bundle["submission"]

try:
    import autosklearn.regression
    from autosklearn.metrics import mean_absolute_error as autosklearn_mae
    AUTO_SKLEARN_AVAILABLE = True
except Exception as exc:
    AUTO_SKLEARN_AVAILABLE = False
    auto_sklearn_import_error = exc

if not AUTO_SKLEARN_AVAILABLE:
    print("auto-sklearn is not installed in the active environment.")
    print("Section 1 skipped cleanly.")
    print("Import error:", repr(auto_sklearn_import_error))
    print("To run this section, install a compatible auto-sklearn build in a separate environment.")
else:
    automl = autosklearn.regression.AutoSklearnRegressor(
        time_left_for_this_task=AUTO_SKLEARN_TIME_LEFT_FOR_THIS_TASK,
        per_run_time_limit=AUTO_SKLEARN_PER_RUN_TIME_LIMIT,
        memory_limit=AUTO_SKLEARN_MEMORY_LIMIT_MB,
        ensemble_size=AUTO_SKLEARN_ENSEMBLE_SIZE,
        metric=autosklearn_mae,
        resampling_strategy="cv",
        resampling_strategy_arguments={"folds": AUTO_SKLEARN_RESAMPLING_FOLDS},
        seed=RANDOM_SEED,
        n_jobs=AUTO_SKLEARN_N_JOBS,
        tmp_folder=AUTO_SKLEARN_TMP_FOLDER,
        delete_tmp_folder_after_terminate=False,
    )

    automl.fit(X_train, y_train, dataset_name="mlb_team_wins")
    train_pred = automl.predict(X_train)
    train_mae = mean_absolute_error(y_train, train_pred)

    pred_values = automl.predict(X_pred)
    sub1 = write_submission(pred_values, submission, AUTO_SKLEARN_OUTPUT_FILE)

    print("Model 1 in-sample MAE:", round(train_mae, 4))
    print("Saved:", AUTO_SKLEARN_OUTPUT_FILE)
    print(sub1.head())

## Section 2 — Model 2: Custom AutoML search over 10 random models

This section runs **10 random model trials one by one**, scores them using cross-validated MAE, and selects the best single model.

### Adjustable hyperparameters
Edit the config block in the next cell:
- `MODEL2_N_TRIALS` — how many random trials to run.
- `MODEL2_CV_SPLITS` — number of CV folds.
- `MODEL2_USE_GROUP_CV` — if `True`, folds are split by `yearID` to reduce era leakage.
- `MODEL2_RANDOM_SEED` — reproducibility.
- `MODEL2_CANDIDATE_POOL` — algorithm families allowed in the search.

### Why this matters
An **MAE** score measures average prediction error in wins.  
Metaphor: if your model predicts 88 wins and the truth is 84, the error is 4. MAE averages that kind of miss across all teams.

In [ ]:
# ============================================================
# MODEL 2 CONFIGURATION — ADJUST THESE HYPERPARAMETERS
# ============================================================

MODEL2_N_TRIALS = 10
MODEL2_CV_SPLITS = 5
MODEL2_USE_GROUP_CV = True
MODEL2_RANDOM_SEED = 42
MODEL2_OUTPUT_FILE = "prediction_submission_2.csv"

# Allowed algorithm families in the random search pool.
MODEL2_CANDIDATE_POOL = [
    "ridge",
    "lasso",
    "elasticnet",
    "random_forest",
    "extra_trees",
    "gradient_boosting",
    "hist_gradient_boosting",
    "lightgbm",
]

In [ ]:
# ============================================================
# MODEL 2 EXECUTION
# ============================================================

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from lightgbm import LGBMRegressor

def sample_model2_estimator(rng):
    algo = rng.choice(MODEL2_CANDIDATE_POOL)

    if algo == "ridge":
        return algo, Ridge(alpha=float(10 ** rng.uniform(-2, 2)), random_state=None)

    if algo == "lasso":
        return algo, Lasso(alpha=float(10 ** rng.uniform(-4, 0)), max_iter=20000, random_state=MODEL2_RANDOM_SEED)

    if algo == "elasticnet":
        return algo, ElasticNet(
            alpha=float(10 ** rng.uniform(-4, 0)),
            l1_ratio=float(rng.uniform(0.05, 0.95)),
            max_iter=20000,
            random_state=MODEL2_RANDOM_SEED
        )

    if algo == "random_forest":
        return algo, RandomForestRegressor(
            n_estimators=int(rng.randint(200, 800)),
            max_depth=int(rng.choice([None, 4, 6, 8, 10, 12])),
            min_samples_split=int(rng.randint(2, 10)),
            min_samples_leaf=int(rng.randint(1, 5)),
            max_features=rng.choice(["sqrt", "log2", 0.5, 0.75, 1.0]),
            random_state=MODEL2_RANDOM_SEED,
            n_jobs=1,
        )

    if algo == "extra_trees":
        return algo, ExtraTreesRegressor(
            n_estimators=int(rng.randint(200, 800)),
            max_depth=int(rng.choice([None, 4, 6, 8, 10, 12])),
            min_samples_split=int(rng.randint(2, 10)),
            min_samples_leaf=int(rng.randint(1, 5)),
            max_features=rng.choice(["sqrt", "log2", 0.5, 0.75, 1.0]),
            random_state=MODEL2_RANDOM_SEED,
            n_jobs=1,
        )

    if algo == "gradient_boosting":
        return algo, GradientBoostingRegressor(
            n_estimators=int(rng.randint(100, 500)),
            learning_rate=float(10 ** rng.uniform(-2.5, -0.3)),
            max_depth=int(rng.randint(2, 5)),
            min_samples_split=int(rng.randint(2, 12)),
            min_samples_leaf=int(rng.randint(1, 6)),
            subsample=float(rng.uniform(0.6, 1.0)),
            random_state=MODEL2_RANDOM_SEED,
        )

    if algo == "hist_gradient_boosting":
        return algo, HistGradientBoostingRegressor(
            learning_rate=float(10 ** rng.uniform(-2.5, -0.3)),
            max_depth=int(rng.choice([None, 3, 4, 5, 6, 8])),
            max_iter=int(rng.randint(150, 600)),
            min_samples_leaf=int(rng.randint(10, 60)),
            l2_regularization=float(10 ** rng.uniform(-5, 1)),
            random_state=MODEL2_RANDOM_SEED,
        )

    if algo == "lightgbm":
        return algo, LGBMRegressor(
            n_estimators=int(rng.randint(200, 900)),
            learning_rate=float(10 ** rng.uniform(-2.5, -0.3)),
            num_leaves=int(rng.randint(15, 80)),
            max_depth=int(rng.choice([-1, 3, 4, 5, 6, 8, 10])),
            min_child_samples=int(rng.randint(5, 60)),
            subsample=float(rng.uniform(0.6, 1.0)),
            colsample_bytree=float(rng.uniform(0.6, 1.0)),
            reg_alpha=float(10 ** rng.uniform(-5, 1)),
            reg_lambda=float(10 ** rng.uniform(-5, 1)),
            random_state=MODEL2_RANDOM_SEED,
            verbosity=-1,
        )

    raise ValueError(f"Unsupported algorithm: {algo}")

data_bundle = prepare_datasets()
X_train = data_bundle["X_train"]
y_train = data_bundle["y_train"]
X_pred = data_bundle["X_pred"]
submission = data_bundle["submission"]
groups = data_bundle["groups"]
numeric_cols = data_bundle["numeric_cols"]
categorical_cols = data_bundle["categorical_cols"]

preprocessor = build_preprocessor(numeric_cols, categorical_cols)
rng = np.random.RandomState(MODEL2_RANDOM_SEED)

trial_results = []
best_pipeline = None
best_mae = np.inf

for trial_idx in range(1, MODEL2_N_TRIALS + 1):
    algo_name, estimator = sample_model2_estimator(rng)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    if MODEL2_USE_GROUP_CV:
        cv_mae_scores = evaluate_model_cv(
            pipeline, X_train, y_train, groups=groups, n_splits=MODEL2_CV_SPLITS
        )
    else:
        cv_mae_scores = evaluate_model_cv(
            pipeline, X_train, y_train, groups=None, n_splits=MODEL2_CV_SPLITS
        )

    cv_mae_mean = float(np.mean(cv_mae_scores))
    cv_mae_std = float(np.std(cv_mae_scores))

    trial_results.append({
        "trial": trial_idx,
        "algorithm": algo_name,
        "cv_mae_mean": cv_mae_mean,
        "cv_mae_std": cv_mae_std,
        "estimator_repr": repr(estimator),
    })

    print(f"Trial {trial_idx:02d} | algo={algo_name:<24} | CV MAE={cv_mae_mean:.4f} ± {cv_mae_std:.4f}")

    if cv_mae_mean < best_mae:
        best_mae = cv_mae_mean
        best_pipeline = clone(pipeline)

trial_results_df = pd.DataFrame(trial_results).sort_values(["cv_mae_mean", "cv_mae_std"]).reset_index(drop=True)
print("\nTop trials:")
print(trial_results_df.head(10))

best_pipeline.fit(X_train, y_train)
pred_values = best_pipeline.predict(X_pred)
sub2 = write_submission(pred_values, submission, MODEL2_OUTPUT_FILE)

print("\nBest CV MAE:", round(best_mae, 4))
print("Saved:", MODEL2_OUTPUT_FILE)
print(sub2.head())

## Section 3 — Model 3: AutoML ensemble from the 5 best models

This section runs a larger random search, keeps the best candidates, then randomly selects **5 models from the best pool** and averages their predictions.

### Adjustable hyperparameters
Edit the config block in the next cell:
- `MODEL3_TOTAL_TRIALS` — how many random models to evaluate.
- `MODEL3_TOP_POOL_SIZE` — how many top models qualify for ensemble selection.
- `MODEL3_ENSEMBLE_SIZE` — number of models sampled into the final ensemble.
- `MODEL3_USE_GROUP_CV` — whether CV is split by `yearID`.
- `MODEL3_RANDOM_SEED` — reproducibility.
- `MODEL3_CANDIDATE_POOL` — allowed algorithms.

### Why this matters
An **ensemble** combines multiple models.  
Metaphor: instead of asking one scout for a forecast, you ask several strong scouts and average them. This often reduces overconfidence and noise.

In [ ]:
# ============================================================
# MODEL 3 CONFIGURATION — ADJUST THESE HYPERPARAMETERS
# ============================================================

MODEL3_TOTAL_TRIALS = 20
MODEL3_TOP_POOL_SIZE = 8
MODEL3_ENSEMBLE_SIZE = 5
MODEL3_CV_SPLITS = 5
MODEL3_USE_GROUP_CV = True
MODEL3_RANDOM_SEED = 123
MODEL3_OUTPUT_FILE = "prediction_submission_3.csv"

MODEL3_CANDIDATE_POOL = [
    "ridge",
    "elasticnet",
    "random_forest",
    "extra_trees",
    "gradient_boosting",
    "hist_gradient_boosting",
    "lightgbm",
]

In [ ]:
# ============================================================
# MODEL 3 EXECUTION
# ============================================================

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from lightgbm import LGBMRegressor

def sample_model3_estimator(rng):
    algo = rng.choice(MODEL3_CANDIDATE_POOL)

    if algo == "ridge":
        return algo, Ridge(alpha=float(10 ** rng.uniform(-2, 2)))

    if algo == "elasticnet":
        return algo, ElasticNet(
            alpha=float(10 ** rng.uniform(-4, 0)),
            l1_ratio=float(rng.uniform(0.05, 0.95)),
            max_iter=20000,
            random_state=MODEL3_RANDOM_SEED
        )

    if algo == "random_forest":
        return algo, RandomForestRegressor(
            n_estimators=int(rng.randint(300, 1000)),
            max_depth=int(rng.choice([None, 5, 7, 9, 12])),
            min_samples_split=int(rng.randint(2, 12)),
            min_samples_leaf=int(rng.randint(1, 6)),
            max_features=rng.choice(["sqrt", "log2", 0.5, 0.75, 1.0]),
            random_state=MODEL3_RANDOM_SEED,
            n_jobs=1,
        )

    if algo == "extra_trees":
        return algo, ExtraTreesRegressor(
            n_estimators=int(rng.randint(300, 1000)),
            max_depth=int(rng.choice([None, 5, 7, 9, 12])),
            min_samples_split=int(rng.randint(2, 12)),
            min_samples_leaf=int(rng.randint(1, 6)),
            max_features=rng.choice(["sqrt", "log2", 0.5, 0.75, 1.0]),
            random_state=MODEL3_RANDOM_SEED,
            n_jobs=1,
        )

    if algo == "gradient_boosting":
        return algo, GradientBoostingRegressor(
            n_estimators=int(rng.randint(150, 700)),
            learning_rate=float(10 ** rng.uniform(-2.5, -0.3)),
            max_depth=int(rng.randint(2, 5)),
            min_samples_split=int(rng.randint(2, 12)),
            min_samples_leaf=int(rng.randint(1, 6)),
            subsample=float(rng.uniform(0.65, 1.0)),
            random_state=MODEL3_RANDOM_SEED,
        )

    if algo == "hist_gradient_boosting":
        return algo, HistGradientBoostingRegressor(
            learning_rate=float(10 ** rng.uniform(-2.5, -0.3)),
            max_depth=int(rng.choice([None, 3, 4, 5, 6, 8])),
            max_iter=int(rng.randint(200, 700)),
            min_samples_leaf=int(rng.randint(10, 80)),
            l2_regularization=float(10 ** rng.uniform(-5, 1)),
            random_state=MODEL3_RANDOM_SEED,
        )

    if algo == "lightgbm":
        return algo, LGBMRegressor(
            n_estimators=int(rng.randint(300, 1200)),
            learning_rate=float(10 ** rng.uniform(-2.5, -0.3)),
            num_leaves=int(rng.randint(15, 100)),
            max_depth=int(rng.choice([-1, 3, 4, 5, 6, 8, 10])),
            min_child_samples=int(rng.randint(5, 80)),
            subsample=float(rng.uniform(0.6, 1.0)),
            colsample_bytree=float(rng.uniform(0.6, 1.0)),
            reg_alpha=float(10 ** rng.uniform(-5, 1)),
            reg_lambda=float(10 ** rng.uniform(-5, 1)),
            random_state=MODEL3_RANDOM_SEED,
            verbosity=-1,
        )

    raise ValueError(f"Unsupported algorithm: {algo}")

data_bundle = prepare_datasets()
X_train = data_bundle["X_train"]
y_train = data_bundle["y_train"]
X_pred = data_bundle["X_pred"]
submission = data_bundle["submission"]
groups = data_bundle["groups"]
numeric_cols = data_bundle["numeric_cols"]
categorical_cols = data_bundle["categorical_cols"]

preprocessor = build_preprocessor(numeric_cols, categorical_cols)
rng = np.random.RandomState(MODEL3_RANDOM_SEED)

candidate_records = []

for trial_idx in range(1, MODEL3_TOTAL_TRIALS + 1):
    algo_name, estimator = sample_model3_estimator(rng)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    if MODEL3_USE_GROUP_CV:
        cv_mae_scores = evaluate_model_cv(
            pipeline, X_train, y_train, groups=groups, n_splits=MODEL3_CV_SPLITS
        )
    else:
        cv_mae_scores = evaluate_model_cv(
            pipeline, X_train, y_train, groups=None, n_splits=MODEL3_CV_SPLITS
        )

    cv_mae_mean = float(np.mean(cv_mae_scores))
    cv_mae_std = float(np.std(cv_mae_scores))

    candidate_records.append({
        "trial": trial_idx,
        "algorithm": algo_name,
        "cv_mae_mean": cv_mae_mean,
        "cv_mae_std": cv_mae_std,
        "pipeline": clone(pipeline),
        "estimator_repr": repr(estimator),
    })

    print(f"Trial {trial_idx:02d} | algo={algo_name:<24} | CV MAE={cv_mae_mean:.4f} ± {cv_mae_std:.4f}")

candidate_records = sorted(candidate_records, key=lambda d: (d["cv_mae_mean"], d["cv_mae_std"]))
top_pool = candidate_records[:MODEL3_TOP_POOL_SIZE]

if len(top_pool) < MODEL3_ENSEMBLE_SIZE:
    raise ValueError(
        f"Top pool size ({len(top_pool)}) is smaller than ensemble size ({MODEL3_ENSEMBLE_SIZE}). "
        "Increase MODEL3_TOTAL_TRIALS or reduce MODEL3_ENSEMBLE_SIZE."
    )

selected_indices = rng.choice(len(top_pool), size=MODEL3_ENSEMBLE_SIZE, replace=False)
selected_models = [top_pool[i] for i in selected_indices]

print("\nSelected ensemble members:")
for item in selected_models:
    print(
        f"trial={item['trial']:02d} | algo={item['algorithm']:<24} | CV MAE={item['cv_mae_mean']:.4f}"
    )

ensemble_predictions = []
for item in selected_models:
    pipe = clone(item["pipeline"])
    pipe.fit(X_train, y_train)
    ensemble_predictions.append(pipe.predict(X_pred))

ensemble_predictions = np.column_stack(ensemble_predictions)
pred_values = ensemble_predictions.mean(axis=1)

sub3 = write_submission(pred_values, submission, MODEL3_OUTPUT_FILE)

selected_maes = [item["cv_mae_mean"] for item in selected_models]
print("\nEnsemble member mean CV MAE:", round(float(np.mean(selected_maes)), 4))
print("Saved:", MODEL3_OUTPUT_FILE)
print(sub3.head())

## Output check

Run the next cell after any section to verify that prediction files exist and match the submission format.

In [ ]:
expected_files = [
    "prediction_submission_1.csv",
    "prediction_submission_2.csv",
    "prediction_submission_3.csv",
]

for file_name in expected_files:
    path = Path(file_name)
    if path.exists():
        tmp = pd.read_csv(path)
        print(f"{file_name}: shape={tmp.shape}, columns={list(tmp.columns)}")
        print(tmp.head(3))
        print("-" * 80)
    else:
        print(f"{file_name}: not created in this run.")